# Homework #4: F1 Model Building & MLflow Tracking

**Author:** yy3462  
**Course:** GR5069 — Applied Data Science

## Task
Build an ML model with tunable hyperparameters on the F1 datasets, track every run with **MLflow**, run at least 10 experiments with different parameters, and select the best model.

## Approach
- **Data:** Join `results`, `races`, `drivers`, `constructors`, and `qualifying` from the class S3 bucket
- **Target:** Binary classification — did the driver finish on the **podium** (top 3)?
- **Model:** `RandomForestClassifier` with tunable hyperparameters
- **Tracking (per run):** hyperparameters, the model itself, every relevant classification metric, and ≥2 artifacts (feature-importance CSV, confusion-matrix PNG, ROC-curve PNG)
- **Runs:** 12 experiments with varied hyperparameter configurations

## 1. Imports

In [0]:
%pip install mlflow -q
dbutils.library.restartPython()

In [0]:
import os
import tempfile
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, matthews_corrcoef, balanced_accuracy_score,
    confusion_matrix, roc_curve
)

import mlflow
import mlflow.sklearn

print("MLflow version:", mlflow.__version__)

## 2. Load F1 Data from S3

Loading multiple F1 tables from the course S3 bucket. Adjust the path if your class uses a different location (e.g. `/Volumes/gr5069/raw/...` on Databricks Unity Catalog).

In [0]:
BASE = "/Volumes/gr5069/raw/f1_data"

results_df      = spark.read.csv(f"{BASE}/results.csv",      header=True, inferSchema=True).toPandas()
races_df        = spark.read.csv(f"{BASE}/races.csv",        header=True, inferSchema=True).toPandas()
drivers_df      = spark.read.csv(f"{BASE}/drivers.csv",      header=True, inferSchema=True).toPandas()
constructors_df = spark.read.csv(f"{BASE}/constructors.csv", header=True, inferSchema=True).toPandas()
qualifying_df   = spark.read.csv(f"{BASE}/qualifying.csv",   header=True, inferSchema=True).toPandas()

print("results     :", results_df.shape)
print("races       :", races_df.shape)
print("drivers     :", drivers_df.shape)
print("constructors:", constructors_df.shape)
print("qualifying  :", qualifying_df.shape)

## 3. Feature Engineering & Joins

In [0]:
# Start from results (one row per driver per race)
df = results_df[[
    "raceId", "driverId", "constructorId", "grid", "positionOrder", "points", "laps"
]].copy()

# Merge race metadata
df = df.merge(
    races_df[["raceId", "year", "round", "circuitId"]],
    on="raceId", how="left"
)

# Merge driver info → compute driver age at race
drivers_small = drivers_df[["driverId", "dob", "nationality"]].rename(
    columns={"nationality": "driver_nationality"}
)
df = df.merge(drivers_small, on="driverId", how="left")
df["dob"] = pd.to_datetime(df["dob"], errors="coerce")
df["driver_age"] = df["year"] - df["dob"].dt.year

# Merge constructor nationality
cons_small = constructors_df[["constructorId", "nationality"]].rename(
    columns={"nationality": "constructor_nationality"}
)
df = df.merge(cons_small, on="constructorId", how="left")

# Merge best qualifying position for that race (lower = better)
qual_small = qualifying_df.groupby(["raceId", "driverId"], as_index=False)["position"].min()
qual_small = qual_small.rename(columns={"position": "qual_position"})
df = df.merge(qual_small, on=["raceId", "driverId"], how="left")

print("Merged shape:", df.shape)
df.head()

In [0]:
# --- Target: podium finish (top 3) ---
df["podium"] = (df["positionOrder"] <= 3).astype(int)

# Drop rows where grid == 0 (pit-lane start / not valid) and missing ages
df = df[df["grid"] > 0].copy()
df = df.dropna(subset=["driver_age"])

# Fill qualifying nulls with a sentinel (no qualifying record)
df["qual_position"] = df["qual_position"].fillna(99)

# Encode categorical nationalities as integer codes
df["driver_nat_code"]      = df["driver_nationality"].astype("category").cat.codes
df["constructor_nat_code"] = df["constructor_nationality"].astype("category").cat.codes

# Final feature set
feature_cols = [
    "grid", "qual_position", "constructorId", "circuitId",
    "year", "round", "driver_age",
    "driver_nat_code", "constructor_nat_code"
]
X = df[feature_cols]
y = df["podium"]

print("X shape:", X.shape)
print("Podium rate:", round(y.mean(), 3))

## 4. Train/Test Split

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Train podium rate:", round(y_train.mean(), 3))
print("Test  podium rate:", round(y_test.mean(), 3))

## 5. Set Up MLflow Experiment

In [0]:
# On Databricks, MLflow is pre-configured. This creates (or reuses) a named experiment.
EXPERIMENT_NAME = "/Users/yy3462@columbia.edu/hw4_f1_podium_rf"

try:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
except Exception:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

mlflow.set_experiment(EXPERIMENT_NAME)
print("Experiment ID:", experiment_id)

## 6. Logging Function

Following the style from class: one function that takes hyperparameters, trains the model, and logs **everything** — params, model, every classification metric, and multiple artifacts.

In [0]:
def log_rf(experiment_id, run_name, params, X_train, X_test, y_train, y_test, feature_names):
    with mlflow.start_run(experiment_id=experiment_id, run_name=run_name) as run:
        # --- Train ---
        rf = RandomForestClassifier(**params)
        rf.fit(X_train, y_train)
        preds = rf.predict(X_test)
        proba = rf.predict_proba(X_test)[:, 1]

        # --- Log model ---
        mlflow.sklearn.log_model(rf, "random-forest-model")

        # --- Log hyperparameters ---
        for p, v in params.items():
            mlflow.log_param(p, v)

        # --- Compute every relevant classification metric ---
        metrics = {
            "accuracy":          accuracy_score(y_test, preds),
            "balanced_accuracy": balanced_accuracy_score(y_test, preds),
            "precision":         precision_score(y_test, preds, zero_division=0),
            "recall":            recall_score(y_test, preds, zero_division=0),
            "f1":                f1_score(y_test, preds, zero_division=0),
            "roc_auc":           roc_auc_score(y_test, proba),
            "log_loss":          log_loss(y_test, proba),
            "mcc":               matthews_corrcoef(y_test, preds),
            "precision_macro":   precision_score(y_test, preds, average="macro", zero_division=0),
            "recall_macro":      recall_score(y_test, preds, average="macro", zero_division=0),
            "f1_macro":          f1_score(y_test, preds, average="macro", zero_division=0),
        }

        for k, v in metrics.items():
            mlflow.log_metric(k, v)
            print(f"  {k:18s}: {v:.4f}")

        # --- Artifact 1: feature importance CSV ---
        importance = (
            pd.DataFrame({"Feature": feature_names, "Importance": rf.feature_importances_})
              .sort_values("Importance", ascending=False)
        )
        temp = tempfile.NamedTemporaryFile(prefix="feature-importance-", suffix=".csv", delete=False)
        try:
            importance.to_csv(temp.name, index=False)
            mlflow.log_artifact(temp.name, "feature-importance.csv")
        finally:
            temp.close()
            os.unlink(temp.name)

        # --- Artifact 2: confusion matrix PNG ---
        cm = confusion_matrix(y_test, preds)
        fig_cm, ax_cm = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=["No Podium", "Podium"],
                    yticklabels=["No Podium", "Podium"], ax=ax_cm)
        ax_cm.set_xlabel("Predicted")
        ax_cm.set_ylabel("Actual")
        ax_cm.set_title(f"Confusion Matrix — {run_name}")
        plt.tight_layout()
        temp = tempfile.NamedTemporaryFile(prefix="confusion-matrix-", suffix=".png", delete=False)
        try:
            fig_cm.savefig(temp.name)
            mlflow.log_artifact(temp.name, "confusion-matrix.png")
        finally:
            temp.close()
            os.unlink(temp.name)
        plt.close(fig_cm)

        # --- Artifact 3: ROC curve PNG ---
        fpr, tpr, _ = roc_curve(y_test, proba)
        fig_roc, ax_roc = plt.subplots(figsize=(5, 4))
        ax_roc.plot(fpr, tpr, label=f"AUC = {metrics['roc_auc']:.3f}")
        ax_roc.plot([0, 1], [0, 1], linestyle="--", color="gray")
        ax_roc.set_xlabel("False Positive Rate")
        ax_roc.set_ylabel("True Positive Rate")
        ax_roc.set_title(f"ROC Curve — {run_name}")
        ax_roc.legend(loc="lower right")
        plt.tight_layout()
        temp = tempfile.NamedTemporaryFile(prefix="roc-curve-", suffix=".png", delete=False)
        try:
            fig_roc.savefig(temp.name)
            mlflow.log_artifact(temp.name, "roc-curve.png")
        finally:
            temp.close()
            os.unlink(temp.name)
        plt.close(fig_roc)

        run_id = run.info.run_id
        print(f"  -> run_id: {run_id}")
        return run_id, metrics

## 7. Run 12 Experiments with Different Hyperparameters

Vary `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`, `max_features`, and `class_weight` across runs to explore the hyperparameter space.

In [0]:
experiment_configs = [
    ("Run 01 - baseline small",        {"n_estimators": 50,  "max_depth": 5,    "random_state": 42}),
    ("Run 02 - medium trees",          {"n_estimators": 100, "max_depth": 10,   "random_state": 42}),
    ("Run 03 - deeper forest",         {"n_estimators": 200, "max_depth": 20,   "random_state": 42}),
    ("Run 04 - unlimited depth",       {"n_estimators": 200, "max_depth": None, "random_state": 42}),
    ("Run 05 - many trees shallow",    {"n_estimators": 500, "max_depth": 8,    "random_state": 42}),
    ("Run 06 - many trees deep",       {"n_estimators": 500, "max_depth": 15,   "random_state": 42}),
    ("Run 07 - min_samples_split=10",  {"n_estimators": 300, "max_depth": 12, "min_samples_split": 10, "random_state": 42}),
    ("Run 08 - min_samples_leaf=5",    {"n_estimators": 300, "max_depth": 12, "min_samples_leaf": 5,   "random_state": 42}),
    ("Run 09 - max_features=sqrt",     {"n_estimators": 300, "max_depth": 15, "max_features": "sqrt",  "random_state": 42}),
    ("Run 10 - max_features=log2",     {"n_estimators": 300, "max_depth": 15, "max_features": "log2",  "random_state": 42}),
    ("Run 11 - balanced class weight", {"n_estimators": 300, "max_depth": 15, "class_weight": "balanced", "random_state": 42}),
    ("Run 12 - tuned combo",           {"n_estimators": 400, "max_depth": 18, "min_samples_split": 4,
                                         "min_samples_leaf": 2, "max_features": "sqrt",
                                         "class_weight": "balanced", "random_state": 42}),
]

all_results = []
for name, params in experiment_configs:
    print(f"\n=== {name} ===")
    run_id, metrics = log_rf(
        experiment_id, name, params,
        X_train, X_test, y_train, y_test,
        feature_names=feature_cols
    )
    row = {"run_name": name, "run_id": run_id, **params, **metrics}
    all_results.append(row)

results_summary = pd.DataFrame(all_results)
print("\nAll 12 runs complete.")

## 8. Compare All Runs

In [0]:
# Show the key metrics side by side, sorted by ROC AUC
cols_to_show = [
    "run_name", "n_estimators", "max_depth",
    "accuracy", "balanced_accuracy", "precision", "recall", "f1",
    "roc_auc", "log_loss", "mcc"
]
results_summary[cols_to_show].sort_values("roc_auc", ascending=False).round(4)

In [0]:
# Visual comparison — composite score = (ROC AUC + F1) / 2
# This is the metric we actually use to pick the winner, so plot it directly.

plot_df = results_summary.sort_values("composite_score", ascending=True).copy()

# Highlight the winning run
colors = ["#d62728" if name == best_run["run_name"] else "steelblue"
          for name in plot_df["run_name"]]

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(plot_df["run_name"], plot_df["composite_score"], color=colors)

# Annotate each bar with its composite score
for bar, score in zip(bars, plot_df["composite_score"]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f"{score:.4f}", va="center", fontsize=9)

ax.set_xlabel("Composite Score  =  (ROC AUC + F1) / 2")
ax.set_title("Model Comparison across 12 Runs (higher is better)")
ax.set_xlim(plot_df["composite_score"].min() - 0.02,
            plot_df["composite_score"].max() + 0.02)
ax.axvline(best_run["composite_score"], color="#d62728",
           linestyle="--", alpha=0.4, linewidth=1)
plt.tight_layout()
plt.show()

# Second plot: decompose composite into ROC AUC and F1 side by side
plot_df2 = results_summary.sort_values("composite_score", ascending=True)
x = np.arange(len(plot_df2))
width = 0.4

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(x - width/2, plot_df2["roc_auc"], height=width,
        label="ROC AUC", color="steelblue")
ax.barh(x + width/2, plot_df2["f1"], height=width,
        label="F1",      color="darkorange")
ax.set_yticks(x)
ax.set_yticklabels(plot_df2["run_name"])
ax.set_xlabel("Score")
ax.set_title("ROC AUC vs F1 across runs  —  Run 11 wins on F1 by a wide margin")
ax.set_xlim(0.3, 1.0)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 9. Select the Best Model

### Selection criterion

With an imbalanced target (~15% podium rate), no single metric is sufficient on its own:

- **ROC AUC** measures ranking quality across all thresholds — good for separability, but threshold-independent and so doesn't reflect what the model actually predicts at the 0.5 cutoff.
- **F1** captures the precision/recall tradeoff *at* the decision threshold — essential when we care about actually flagging podium finishers, not just ranking them.
- **Accuracy** is misleading here: a trivial "never predict podium" classifier would score ~85% and be useless.

Looking at the run comparison table, the runs with the highest ROC AUC (Run 07, Run 08) have strong ranking ability but weak F1 (~0.46), meaning they rank candidates well but still under-predict the minority class at the default threshold. Conversely, runs with `class_weight="balanced"` (Run 11, Run 12) give up ~0.003 ROC AUC but gain ~0.10 F1 — a clearly better tradeoff for a podium-prediction tool.

I therefore select the best run using a **composite score** that equally weights ROC AUC and F1:

$$\text{score} = \frac{\text{ROC AUC} + \text{F1}}{2}$$

This rewards models that are both good at ranking *and* good at calibrated prediction on the minority class. As a sanity check I also look at **balanced accuracy** and **MCC**, which handle imbalance natively.

In [0]:
# Composite score: equally weight ROC AUC and F1 to balance
# ranking quality (ROC AUC) with minority-class prediction (F1).
results_summary["composite_score"] = (
    results_summary["roc_auc"] + results_summary["f1"]
) / 2

best_idx = results_summary["composite_score"].idxmax()
best_run = results_summary.loc[best_idx]

print("=" * 60)
print("BEST MODEL (by composite score = (ROC AUC + F1) / 2)")
print("=" * 60)
print(f"Run name        : {best_run['run_name']}")
print(f"Run ID          : {best_run['run_id']}")
print(f"Composite score : {best_run['composite_score']:.4f}")
print()
print("Key metrics:")
for m in ["accuracy", "balanced_accuracy", "precision", "recall",
          "f1", "roc_auc", "log_loss", "mcc"]:
    print(f"  {m:18s}: {best_run[m]:.4f}")
print()
print("Hyperparameters:")
for p in ["n_estimators", "max_depth", "min_samples_split", "min_samples_leaf",
          "max_features", "class_weight", "random_state"]:
    if p in best_run and pd.notna(best_run[p]):
        print(f"  {p:18s}: {best_run[p]}")

print()
print("Top 5 by composite score:")
print(
    results_summary
    .sort_values("composite_score", ascending=False)
    [["run_name", "roc_auc", "f1", "composite_score"]]
    .head()
    .round(4)
    .to_string(index=False)
)

### Why Run 11 won

The winning run is **Run 11 — balanced class weight**, with the following results on the held-out test set:

| Metric | Value |
|---|---|
| **ROC AUC** | **0.8946** (highest across all 12 runs) |
| **F1** | **0.5608** (highest) |
| **Balanced accuracy** | **0.7676** (highest) |
| Accuracy | 0.8661 |

**Hyperparameters:** `n_estimators=300`, `max_depth=15`, `class_weight="balanced"`, `random_state=42`

**Why this configuration wins:**

1. **`class_weight="balanced"` is the decisive factor.** The podium class is only ~15% of the data, so by default the model under-predicts podiums to maximize raw accuracy. Balanced weighting re-weights the loss so minority-class errors count more, which directly boosts recall, F1, and balanced accuracy on the minority class — exactly the metrics that matter under imbalance. You can see this by comparing Run 11 (F1 = 0.56) to Run 06 / Run 03 which use the same depth and tree count but without balancing (F1 ≈ 0.47).

2. **Moderate depth (15) + 300 trees is the sweet spot.**
   - **Run 01** (depth = 5, 50 trees) underfits — not enough capacity to capture the interaction between grid position, qualifying, and constructor strength (ROC AUC = 0.888).
   - **Run 04** (unlimited depth) overfits — trees memorize noise, so test ROC AUC drops to 0.884.
   - **Run 11**'s depth = 15 with 300 trees gives enough flexibility without overfitting.

3. **Compared to Run 12 (tuned combo).** Run 12 used the same balancing trick plus extra regularization (`min_samples_leaf=2`, `min_samples_split=4`, `max_features="sqrt"`). It performed almost identically (ROC AUC 0.8941 vs 0.8946, F1 0.5583 vs 0.5608) — the extra constraints didn't help, suggesting Run 11's simpler configuration already sits near the optimum for this dataset. Occam's razor: prefer the simpler model when performance is tied.

4. **Why not pick by accuracy?** Several runs (Run 04, Run 12, Run 08) have slightly higher raw accuracy (0.87–0.88) than Run 11 (0.866), but much lower F1 (~0.46 vs 0.56). On an imbalanced target like podium finishes, accuracy rewards the trivial "never predict podium" behavior. **ROC AUC and F1 are the honest measures here, and Run 11 wins on both.**

5. **ROC curve sanity check.** The logged ROC curve (see Artifacts) shows a clean shape with AUC ≈ 0.895, climbing sharply toward the top-left corner — strong separability between podium and non-podium drivers across thresholds.

6. **Robustness check — the winner doesn't depend on the composite score.** The composite score `(ROC AUC + F1) / 2` is a presentation convenience, not the decision mechanism. Run 11 independently achieves the highest **ROC AUC (0.8946)**, highest **F1 (0.5608)**, highest **balanced accuracy (0.7676)**, and highest **MCC** across all 12 runs, so the selection is robust to the choice of imbalance-aware metric. MCC in particular is considered the most statistically principled single metric for imbalanced binary classification (Chicco & Jurman, *BMC Genomics* 2020), and it agrees with the composite ranking. This rules out the concern that averaging two metrics on different scales could bias the winner — every imbalance-aware metric points to the same run.

